# 03 · Aprendizaje no supervisado: tipos de día
**TráficoIA · Proyecto Final, versión 3 · Semana 2 (lunes 23 de noviembre)**

Equipo N.º ____ · Integrantes: ______________________

**Pregunta:** si no le decimos al algoritmo qué días son laborales, de fin de semana o festivos, ¿los descubre solo a partir de la forma de su curva de tráfico? Y si lo hace, ¿qué horario de semáforo conviene a cada tipo de día?

**Método:**
1. Cada día se convierte en un punto de **24 valores** (volumen promedio en cada hora).
2. Se escala con `StandardScaler` (obligatorio antes de agrupar, para que ninguna hora pese más por tener números más grandes).
3. **K-Means**: se elige el número de grupos con el método del codo y la silueta.
4. **DBSCAN**: agrupa por densidad y marca como *ruido* los días raros. Se elige `eps` con la gráfica de distancias.
5. **PCA** solo para dibujar en 2D; los grupos se calculan con las 24 dimensiones.
6. Se interpreta cada grupo y se propone un horario de semáforo para cada tipo de día.

> **Antes de ejecutar:** haber descargado los datos (`python src/descargar_datos.py --tabular`).

In [ ]:
import sys, warnings
from pathlib import Path

# Permite importar src/ aunque el cuaderno se ejecute desde notebooks/
RAIZ = Path.cwd().resolve()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config, no_supervisado

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
config.FIGURAS.mkdir(parents=True, exist_ok=True)

def guardar(nombre):
    plt.tight_layout()
    plt.savefig(config.FIGURAS / f"{nombre}.png", dpi=130, bbox_inches="tight")

print("Raíz del proyecto:", RAIZ)

from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score

## 1. De horas sueltas a perfiles diarios

In [ ]:
df = no_supervisado.cargar_datos_limpios()
perfiles = no_supervisado.matriz_perfiles(df, max_faltantes=3)
meta = no_supervisado.metadatos_dias(df)

print(f"Días en los datos: {df['fecha'].nunique():,}")
print(f"Días utilizables (hasta 3 horas faltantes, interpoladas): {len(perfiles):,}")
print(f"Matriz de perfiles: {perfiles.shape[0]} días x {perfiles.shape[1]} horas")

fig, ax = plt.subplots(figsize=(10, 4))
for _, fila in perfiles.sample(min(80, len(perfiles)), random_state=config.SEMILLA).iterrows():
    ax.plot(range(24), fila.values, alpha=0.15, color="#2a6f97")
ax.plot(range(24), perfiles.mean().values, color="k", lw=2.5, label="Promedio de todos los días")
ax.set(title="Perfiles horarios de 80 días al azar", xlabel="Hora", ylabel="Vehículos por hora", xticks=range(0, 24, 2))
ax.legend()
guardar("nosup_perfiles_diarios")
plt.show()

*Interpretación:* ¿se ven formas de día distintas a simple vista (con dos picos y con uno solo suave)? Esa estructura es lo que los algoritmos intentarán descubrir. Anota cuántos días se descartaron por tener demasiadas horas faltantes y si eso puede sesgar el resultado (por ejemplo, si faltan sobre todo días de una temporada).

## 2. Escalar las variables

In [ ]:
escalador, X = no_supervisado.escalar(perfiles)
print("Forma de X:", X.shape)
print("Media por columna tras escalar (≈ 0):", np.round(X.mean(axis=0)[:5], 3), "...")
print("Desviación por columna tras escalar (≈ 1):", np.round(X.std(axis=0)[:5], 3), "...")

## 3. K-Means: ¿cuántos grupos?
Dos criterios, que se leen juntos:
* **Codo:** la inercia (suma de distancias al centro) siempre baja al aumentar k; se busca el punto donde la mejora se vuelve pequeña.
* **Silueta:** entre −1 y 1, mide qué tan separados están los grupos. Más alta es mejor.

In [ ]:
ev = no_supervisado.evaluar_k(X, ks=range(2, 11))
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
ejes[0].plot(ev["k"], ev["inercia"], marker="o")
ejes[0].set(title="Método del codo", xlabel="Número de grupos (k)", ylabel="Inercia")
ejes[1].plot(ev["k"], ev["silueta"], marker="o", color="#d62828")
ejes[1].set(title="Coeficiente de silueta", xlabel="Número de grupos (k)", ylabel="Silueta")
for ax in ejes:
    ax.set_xticks(range(2, 11))
guardar("nosup_codo_silueta")
plt.show()
ev.round(3)

**Decisión de k (escríbela y justifícala).** El código propone el k con mayor silueta, pero **tú decides**: la silueta suele favorecer k = 2 (laboral frente a fin de semana), mientras que el codo o el sentido del problema pueden pedir 3 o más (por ejemplo, separar festivos o viernes). Cambia `K_ELEGIDO` si tu justificación apunta a otro valor.

In [ ]:
K_ELEGIDO = None        # escribe un entero (por ejemplo 3) para forzar tu elección; None = el de mayor silueta

K = K_ELEGIDO or no_supervisado.k_por_silueta(ev)
print("k usado:", K, "(propuesto por silueta)" if K_ELEGIDO is None else "(elegido por el equipo)")
km = no_supervisado.kmeans_final(X, K)
print("Silueta con ese k:", round(float(ev.loc[ev['k'] == K, 'silueta'].iloc[0]), 3))

## 4. Visualizar con PCA

In [ ]:
pca = PCA(n_components=2, random_state=config.SEMILLA)
Z = pca.fit_transform(X)
print("Varianza explicada por PC1 y PC2:", np.round(pca.explained_variance_ratio_, 3),
      "| total:", round(float(pca.explained_variance_ratio_.sum()), 3))

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(Z[:, 0], Z[:, 1], c=km.labels_, cmap="tab10", s=14, alpha=0.7)
ax.set(title=f"K-Means (k = {K}) en el plano de las 2 primeras componentes",
       xlabel=f"PC1 ({pca.explained_variance_ratio_[0]:.0%})", ylabel=f"PC2 ({pca.explained_variance_ratio_[1]:.0%})")
ax.legend(*sc.legend_elements(), title="Grupo", loc="best")
guardar("nosup_pca_kmeans")
plt.show()

*Interpretación:* si PC1 y PC2 explican poca varianza, la imagen en 2D es una simplificación y dos grupos que se ven mezclados pueden estar separados en las otras dimensiones. ¿Los grupos se ven separados o se tocan? Una nube continua sin huecos indica que los tipos de día no son fronteras nítidas sino un gradiente.

## 5. Interpretar cada grupo en términos del problema
Los algoritmos no saben qué es un «laboral». Para ponerles nombre se compara cada grupo con información que el algoritmo **no usó**: día de la semana y festivo.

In [ ]:
resumen = no_supervisado.describir_clusters(perfiles, km.labels_, meta)
resumen.round(2)

In [ ]:
nombres_auto = no_supervisado.nombrar_clusters(resumen)
print("Nombres automáticos de partida:", nombres_auto)

# Corrige aquí los nombres si, tras mirar las gráficas, algún grupo merece otro (ejemplo: {0: "Laboral", 1: "Fin de semana"})
NOMBRES_MANUALES = {}
nombres = {**nombres_auto, **NOMBRES_MANUALES}
etiqueta = pd.Series(km.labels_, index=perfiles.index).map(nombres).rename("tipo_de_dia")
etiqueta.value_counts().to_frame("n_dias")

In [ ]:
colores = sns.color_palette("tab10", K)
fig, ejes = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1.3, 1]})
for g, color in zip(sorted(nombres), colores):
    dias_g = perfiles[km.labels_ == g]
    media, desv = dias_g.mean(), dias_g.std()
    ejes[0].plot(range(24), media.values, color=color, lw=2.4, label=f"{nombres[g]} (n = {len(dias_g)})")
    ejes[0].fill_between(range(24), (media - desv).values, (media + desv).values, color=color, alpha=0.15)
ejes[0].set(title="Perfil promedio de cada tipo de día (banda = ± 1 desviación)", xlabel="Hora",
            ylabel="Vehículos por hora", xticks=range(0, 24, 2))
ejes[0].legend(fontsize=9)

comp = pd.crosstab(etiqueta, meta.reindex(perfiles.index)["dia_semana"].map(dict(enumerate(no_supervisado.DIAS))),
                   normalize="index").reindex(columns=no_supervisado.DIAS)
sns.heatmap(comp, annot=True, fmt=".0%", cmap="Blues", ax=ejes[1], cbar=False)
ejes[1].set(title="¿Qué días de la semana hay en cada grupo?", xlabel="", ylabel="")
guardar("nosup_perfiles_por_tipo")
plt.show()

*Interpretación (escríbela con tus datos):*
* ¿Qué grupo es el laboral? ¿Cuál es el fin de semana? Fíjate en el porcentaje de sábados y domingos de cada uno y en la forma: el laboral tiene dos picos marcados (hora de entrada y de salida), el de fin de semana un solo lomo suave a mediodía.
* ¿Los festivos caen en el grupo de fin de semana, en el laboral, o repartidos? Revisa la columna `pct_festivo` de la tabla. Un festivo que se comporta como domingo es un resultado esperable; uno que se comporta como laboral es un hallazgo interesante.
* Si hay más de un grupo laboral, ¿qué los distingue (nivel de volumen, hora del pico, viernes frente a lunes)?

## 6. DBSCAN: agrupar por densidad y detectar días raros
DBSCAN necesita dos parámetros: `min_samples` (vecinos mínimos para formar un grupo) y `eps` (radio). El `eps` se elige mirando la gráfica de distancias al k-ésimo vecino: el «codo» marca el punto donde empiezan los días aislados.

In [ ]:
MIN_SAMPLES = 5
kdist = no_supervisado.distancias_kvecinos(X, MIN_SAMPLES)
percentiles = [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]
eps_candidatos = np.quantile(kdist, percentiles)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(kdist)
for p, e in zip(percentiles, eps_candidatos):
    ax.axhline(e, color="gray", ls=":", lw=0.8)
    ax.text(len(kdist) * 0.02, e, f"p{int(p*100)} = {e:.2f}", fontsize=8, va="bottom")
ax.set(title=f"Distancia al vecino n.º {MIN_SAMPLES} (ordenada)", xlabel="Días ordenados", ylabel="Distancia")
guardar("nosup_dbscan_kdist")
plt.show()

In [ ]:
tabla_db = no_supervisado.evaluar_dbscan(X, eps_candidatos, MIN_SAMPLES)
tabla_db.round(3)

*Interpretación:* a `eps` pequeño casi todo es ruido; a `eps` grande todo se junta en un solo grupo. Se busca un punto intermedio con al menos 2 grupos y poco ruido. La regla del código (mayor silueta con ≤ 15 % de ruido) es un punto de partida: cámbiala si tu criterio es otro y explícalo.

In [ ]:
EPS_ELEGIDO = None      # escribe un número (por ejemplo 3.6) para forzar tu elección; None = regla automática

fila = no_supervisado.elegir_eps(tabla_db)
EPS = EPS_ELEGIDO or float(fila["eps"])
from sklearn.cluster import DBSCAN
db = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES).fit(X)
et_db = db.labels_
n_g, ruido = len(set(et_db) - {-1}), (et_db == -1).mean()
print(f"eps usado: {EPS:.3f} | grupos: {n_g} | ruido: {ruido:.1%} ({(et_db == -1).sum()} días)")

fig, ax = plt.subplots(figsize=(8, 6))
m = et_db == -1
ax.scatter(Z[~m, 0], Z[~m, 1], c=et_db[~m], cmap="tab10", s=14, alpha=0.7)
ax.scatter(Z[m, 0], Z[m, 1], c="black", marker="x", s=30, label=f"Ruido ({m.sum()} días)")
ax.set(title=f"DBSCAN (eps = {EPS:.2f}, min_samples = {MIN_SAMPLES}) en el plano PCA", xlabel="PC1", ylabel="PC2")
ax.legend()
guardar("nosup_pca_dbscan")
plt.show()

In [ ]:
# ¿Qué días marcó DBSCAN como ruido?
ruido_dias = meta.reindex(perfiles.index)[et_db == -1].copy()
ruido_dias["dia"] = ruido_dias["dia_semana"].map(dict(enumerate(no_supervisado.DIAS)))
ruido_dias["volumen_medio"] = perfiles[et_db == -1].mean(axis=1).round(0)
ruido_dias[["dia", "festivo", "volumen_medio"]].head(25)

*Interpretación:* los días de ruido suelen ser **tormentas, accidentes, festivos puente o fallas de sensor**: curvas que no se parecen a ningún tipo habitual. ¿Qué tienen en común los que ves aquí (día de la semana, festivo, volumen muy bajo)? Ese es el valor propio de DBSCAN frente a K-Means, que obliga a cada día a pertenecer a algún grupo.

## 7. Comparar K-Means y DBSCAN

In [ ]:
sin_ruido = et_db != -1
ari = adjusted_rand_score(km.labels_[sin_ruido], et_db[sin_ruido]) if n_g >= 1 and sin_ruido.sum() > 1 else float("nan")
from sklearn.metrics import silhouette_score
comparacion = pd.DataFrame({
    "K-Means": {"grupos": K, "ruido": 0.0, "silueta": silhouette_score(X, km.labels_)},
    "DBSCAN": {"grupos": n_g, "ruido": ruido,
               "silueta": silhouette_score(X[sin_ruido], et_db[sin_ruido]) if n_g >= 2 else float("nan")},
}).T
print(f"Acuerdo entre ambos en los días que DBSCAN agrupó (índice de Rand ajustado): {ari:.3f}  (1 = idénticos)")
comparacion.round(3)

*Interpretación:* la silueta de DBSCAN se calcula sin los días de ruido, así que **no es directamente comparable** con la de K-Means (que clasifica todos). Comenta qué aporta cada método: K-Means da una partición completa y estable; DBSCAN da un número de grupos que descubre solo y señala los días anómalos.

## 8. Propuesta de horario de semáforo por tipo de día
Idea: cuando el tráfico es alto, el semáforo necesita un ciclo más largo (menos cambios de fase, más autos que pasan por verde); cuando es bajo, un ciclo corto evita esperas innecesarias. Se convierte el perfil de cada grupo en franjas con un ciclo sugerido.

> **Supuestos de partida:** ciclos de 45 s (bajo), 70 s (medio) y 100 s (alto), y niveles por terciles de todo el volumen horario. Son **hipótesis**, no resultados: la semana 3 las pondrá a prueba con el simulador de refuerzo.

In [ ]:
cortes = no_supervisado.cortes_globales(perfiles)
print("Cortes de nivel horario: bajo <= {:.0f} < medio <= {:.0f} < alto".format(*cortes))

propuestas = {}
for g in sorted(nombres):
    perfil_g = perfiles[km.labels_ == g].mean()
    tabla_g = no_supervisado.propuesta_semaforo(perfil_g, cortes)
    propuestas[nombres[g]] = tabla_g
    print(f"\n=== {nombres[g]} ===")
    display(tabla_g)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
for g, color in zip(sorted(nombres), colores):
    perfil_g = perfiles[km.labels_ == g].mean()
    ciclo = [no_supervisado.CICLO_POR_NIVEL[no_supervisado._nivel(v, cortes)] for v in perfil_g.values]
    ax.step(range(24), ciclo, where="post", color=color, lw=2.2, label=nombres[g])
ax.set(title="Ciclo de semáforo sugerido por hora y tipo de día", xlabel="Hora", ylabel="Ciclo (s)",
       xticks=range(0, 24, 2), yticks=sorted(no_supervisado.CICLO_POR_NIVEL.values()))
ax.legend()
guardar("nosup_semaforo_propuesto")
plt.show()

*Interpretación:* ¿las franjas tienen sentido operativo (no cambia el ciclo cada hora)? ¿Qué diferencia práctica hay entre el horario laboral y el de fin de semana? ¿Qué habría que medir para validar estos ciclos?

## 9. Guardar los modelos y los resultados

In [ ]:
paquete_km = {"version": "semana2", "scaler": escalador, "kmeans": km, "k": K, "nombres": nombres,
              "horas": list(range(24)), "cortes_nivel": cortes,
              "propuesta_semaforo": {n: t.to_dict("records") for n, t in propuestas.items()},
              "silueta": float(silhouette_score(X, km.labels_))}
no_supervisado.guardar(paquete_km, no_supervisado.RUTA_KMEANS)

paquete_db = {"version": "semana2", "scaler": escalador, "eps": EPS, "min_samples": MIN_SAMPLES,
              "n_grupos": n_g, "proporcion_ruido": float(ruido)}
no_supervisado.guardar(paquete_db, no_supervisado.RUTA_DBSCAN)

# Tablas para el informe (pequeñas, se suben a GitHub)
reportes = config.RAIZ / "reports"
resumen.assign(tipo_de_dia=pd.Series(nombres)).round(3).to_csv(reportes / "tabla_clusters.csv")
pd.concat([t.assign(tipo_de_dia=n) for n, t in propuestas.items()]).to_csv(reportes / "propuesta_semaforo.csv", index=False)

# Etiqueta de cada día (grande, queda en data/processed y NO se sube)
config.DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"tipo_kmeans": etiqueta, "grupo_dbscan": et_db}, index=perfiles.index) \
  .to_csv(config.DATA_PROCESSED / "dias_clusters.csv")
print("Guardado en models/ y reports/")

In [ ]:
# Prueba de uso: así clasificará la app un día nuevo
paq = no_supervisado.cargar_kmeans()
for g in sorted(paq["nombres"]):
    ejemplo = perfiles[km.labels_ == g].iloc[0].values
    print(f"Día real del grupo {g}:", no_supervisado.clasificar_dia(paq, ejemplo))

## Conclusiones del módulo no supervisado
*Completa con 4 a 6 hallazgos concretos y con números de TU ejecución:*

1. Cuántos tipos de día encontraste (k) y por qué elegiste ese número (codo y silueta).
2. Qué es cada grupo y cómo lo comprobaste (día de la semana, festivo, forma de la curva).
3. Qué días marcó DBSCAN como ruido y qué tienen en común.
4. Qué aporta cada método frente al otro.
5. El horario de semáforo propuesto para cada tipo de día y los supuestos detrás.
6. Limitaciones: una sola autopista, días descartados por horas faltantes, escalado por hora (se pierde el nivel absoluto de cada día).